### Conexión AstraDB y Spark session

In [14]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/big-data-final"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Instalar Java 17
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

# Instalar PySpark y driver
!pip install pyspark==3.5.0 cassandra-driver -q

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

print("Dependencias instaladas.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 20.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires pyspark[connect]~=3.5.1, but you have pyspark 3.5.0 which is incompatible.
Dependencias instaladas.


In [15]:
# Rutas del Data Lake
LANDING_PATH = f"{PROJECT_ROOT}/datalake/landing"
BRONZE_PATH = f"{PROJECT_ROOT}/datalake/bronze"
SILVER_PATH = f"{PROJECT_ROOT}/datalake/silver"
GOLD_PATH = f"{PROJECT_ROOT}/datalake/gold"
CHECKPOINT_PATH = f"{PROJECT_ROOT}/datalake/checkpoints"
QUARANTINE_PATH = f"{PROJECT_ROOT}/datalake/quarantine"

# Crear estructura de directorios si no existe
os.makedirs(LANDING_PATH, exist_ok=True)
os.makedirs(BRONZE_PATH, exist_ok=True)
os.makedirs(SILVER_PATH, exist_ok=True)
os.makedirs(GOLD_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(QUARANTINE_PATH, exist_ok=True)

print(f"Directorios configurados en: {PROJECT_ROOT}")

Directorios configurados en: /content/drive/MyDrive/big-data-final


In [34]:
import shutil
from google.colab import userdata

# Credenciales de AstraDB
ASTRA_CLIENT_ID = userdata.get('ASTRA_CLIENT_ID')
ASTRA_CLIENT_SECRET = userdata.get('ASTRA_CLIENT_SECRET')

# Ruta al Secure Connect Bundle (SCB)
SCB_PATH = f"{PROJECT_ROOT}/secure-connect-cloud-analytics.zip"

if os.path.exists(SCB_PATH):
    print(f"Secure Connect Bundle encontrado en: {SCB_PATH}")
    abs_path = os.path.abspath(SCB_PATH)
    SCB_URI = f"file://{abs_path}"
else:
    print(f"No se encuentra el Secure Connect Bundle en {SCB_PATH}")
    raise FileNotFoundError(f"Sube el secure-connect-bundle.zip a {PROJECT_ROOT}")

if 'spark' in locals():
    spark.stop()

Secure Connect Bundle encontrado en: /content/drive/MyDrive/big-data-final/secure-connect-cloud-analytics.zip


In [35]:
# SparkSession
# Configuramos Spark para que descargue automáticamente el conector de Cassandra y utilice el SCB.
from pyspark.sql import SparkSession

SPARK_PACKAGES = "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0"

spark = SparkSession.builder \
    .appName("CloudProviderAnalytics_Colab") \
    .master("local[*]") \
    .config("spark.jars.packages", SPARK_PACKAGES) \
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions") \
    .config("spark.sql.catalog.myCatalog", "com.datastax.spark.connector.datasource.CassandraCatalog") \
    .config("spark.cassandra.connection.config.cloud.path", SCB_URI) \
    .config("spark.cassandra.auth.username", ASTRA_CLIENT_ID) \
    .config("spark.cassandra.auth.password", ASTRA_CLIENT_SECRET) \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"Spark Session iniciada. Versión: {spark.version}")

Spark Session iniciada. Versión: 3.5.0


In [44]:
# Smoke test para verificar la conexión

# 1. Test Spark Local
try:
    print("Test 1: Spark Local DataFrame...")
    spark.range(3).show()
    print("✅ Spark local OK.")
except Exception as e:
    print(f"❌ Error Spark local: {e}")

# 2. Test Conectividad AstraDB
print("\nTest 2: Conexión a AstraDB...")
try:
    # Usamos el catálogo 'myCatalog' que definimos en la configuración
    # Esto le pide a AstraDB la lista de Keyspaces visibles
    spark.sql("SHOW NAMESPACES IN myCatalog").show()
    print("✅ Verifica la existencia de 'cloud_analytics'")
    spark.sql("DESCRIBE NAMESPACE myCatalog.cloud_analytics").show(truncate=False)

except Exception as e:
    print(f"❌ Error al listar bases de datos: {e}")

Test 1: Spark Local DataFrame...
+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+

✅ Spark local OK.

Test 2: Conexión a AstraDB...
+------------------+
|         namespace|
+------------------+
|   cloud_analytics|
|data_endpoint_auth|
|      datastax_sla|
+------------------+

✅ Verifica la existencia de 'cloud_analytics'
+--------------+---------------+
|info_name     |info_value     |
+--------------+---------------+
|Catalog Name  |myCatalog      |
|Namespace Name|cloud_analytics|
+--------------+---------------+



## Ingest Batch (Landing -> Bronze)

In [45]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, input_file_name

print("Ingest Batch a Bronze...")

# DEFINICIÓN DE ESQUEMAS
# Definimos los tipos manualmente para asegurar calidad desde el inicio.
# Esto evita que Spark adivine mal (ej. tratar un ID numérico como entero cuando debería ser string)

schemas = {
    "customers_orgs": StructType([
        StructField("org_id", StringType(), True),
        StructField("org_name", StringType(), True),
        StructField("industry", StringType(), True),
        StructField("country", StringType(), True),
        StructField("region", StringType(), True),
        StructField("subscription_plan", StringType(), True),
        StructField("creation_date", DateType(), True) # [cite: 18]
    ]),
    "users": StructType([
        StructField("user_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("email", StringType(), True),
        StructField("role", StringType(), True),
        StructField("status", StringType(), True),
        StructField("last_login", TimestampType(), True) # [cite: 19]
    ]),
    "resources": StructType([
        StructField("resource_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("service", StringType(), True),
        StructField("region", StringType(), True),
        StructField("type", StringType(), True),
        StructField("tags", StringType(), True),
        StructField("creation_time", TimestampType(), True) # [cite: 20]
    ]),
    "billing_monthly": StructType([
        StructField("org_id", StringType(), True),
        StructField("billing_month", StringType(), True), # Formato YYYY-MM
        StructField("currency", StringType(), True),
        StructField("tax_rate", DoubleType(), True),
        StructField("credits_applied", DoubleType(), True),
        StructField("total_due", DoubleType(), True) # [cite: 30]
    ]),
    "support_tickets": StructType([
        StructField("ticket_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("user_id", StringType(), True),
        StructField("severity", StringType(), True),
        StructField("status", StringType(), True),
        StructField("category", StringType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("resolved_at", TimestampType(), True),
        StructField("sla_due", TimestampType(), True),
        StructField("csat_score", IntegerType(), True) # [cite: 21]
    ]),
    "marketing_touches": StructType([
        StructField("touch_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("campaign_id", StringType(), True),
        StructField("channel", StringType(), True),
        StructField("touch_timestamp", TimestampType(), True),
        StructField("converted", BooleanType(), True) # [cite: 22]
    ]),
    "nps_surveys": StructType([
        StructField("survey_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("user_id", StringType(), True),
        StructField("survey_date", DateType(), True),
        StructField("score", IntegerType(), True),
        StructField("comment", StringType(), True) # [cite: 29]
    ])
}

# FUNCIÓN DE INGEST ESTÁNDAR
def ingest_batch_file(file_name, schema):
    source_path = f"{LANDING_PATH}/{file_name}.csv"
    dest_path = f"{BRONZE_PATH}/{file_name}"

    print(f"Procesando: {file_name}...")
    try:
        # mode="PERMISSIVE": Si una fila está muy mal formada, pone nulls pero no rompe el proceso
        df = spark.read.csv(source_path, header=True, schema=schema, mode="PERMISSIVE")

        # Agregar marcas de ingesta (Requisito Bronze)
        df_bronze = df \
            .withColumn("ingest_ts", current_timestamp()) \
            .withColumn("source_file", input_file_name())

        # Escribir a Parquet
        df_bronze.write.mode("overwrite").parquet(dest_path)

        count = df_bronze.count()
        print(f"Guardado en Bronze: {dest_path}")
        print(f"Registros procesados: {count}")

    except Exception as e:
        print(f"Error crítico en {file_name}: {e}")

# EJECUCIÓN DEL PIPELINE BATCH
for name, schema in schemas.items():
    ingest_batch_file(name, schema)

print("\nCapa Bronze Batch lista.")

Ingest Batch a Bronze...
Procesando: customers_orgs...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/customers_orgs
Registros procesados: 80
Procesando: users...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/users
Registros procesados: 800
Procesando: resources...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/resources
Registros procesados: 400
Procesando: billing_monthly...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/billing_monthly
Registros procesados: 240
Procesando: support_tickets...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/support_tickets
Registros procesados: 1000
Procesando: marketing_touches...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/marketing_touches
Registros procesados: 1500
Procesando: nps_surveys...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/nps_surveys
Registros

In [46]:
print("Verificando tabla 'resources' en Bronze:")
df_check = spark.read.parquet(f"{BRONZE_PATH}/resources")
df_check.show(5)
df_check.printSchema()

Verificando tabla 'resources' en Bronze:
+------------+------------+----------+----------+----------+-------+-------------+--------------------+--------------------+
| resource_id|      org_id|   service|    region|      type|   tags|creation_time|           ingest_ts|         source_file|
+------------+------------+----------+----------+----------+-------+-------------+--------------------+--------------------+
|res_eubfn9kr|org_pnsm43d8|   compute|   sa-east|2025-08-14|running|         NULL|2025-12-03 18:50:...|file:///content/d...|
|res_fvb66h3r|org_i7p5tb94|  database|  ap-south|2025-06-05|stopped|         NULL|2025-12-03 18:50:...|file:///content/d...|
|res_cbrlqmn4|org_d14ve92m|   storage|eu-central|2025-08-10|running|         NULL|2025-12-03 18:50:...|file:///content/d...|
|res_ew1yf0dw|org_pja1wj0t|networking|   us-west|2025-06-02|running|         NULL|2025-12-03 18:50:...|file:///content/d...|
|res_n6mbypjd|org_pja1wj0t|   storage|   us-west|2025-06-05|running|         NULL|20